In [5]:
import pandas as pd
import glob
import re
from pathlib import Path
from typing import List


def load_all_csvs(input_dir: str) -> pd.DataFrame:
    pattern = f"{input_dir}/*.[cC][sS][vV]"
    files = glob.glob(pattern)
    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f, delimiter=";", low_memory=False))
        except Exception as e:
            print(f"Failed to read {f}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def filter_by(
    df: pd.DataFrame, values: List[str], column_name: str = None
) -> pd.DataFrame:
    if df.empty:
        return df
    pattern = "|".join([re.escape(value) for value in values])
    if column_name and column_name in df.columns:
        mask = df[column_name].astype(str).str.contains(pattern, case=False, na=False)
    else:
        return pd.DataFrame()
    return df[mask].copy()


def select_columns(df: pd.DataFrame, desired_columns: List[str]) -> pd.DataFrame:
    present = [c for c in desired_columns if c in df.columns]
    return df[present].copy()


def save(
    data: pd.DataFrame,
    output_path: str,
) -> pd.DataFrame:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(output_path, index=False, sep=";")
    print(f"Saved {len(df_out)} rows to {output_path}")


desired_columns = [
    "LONGITUDE",
    "LATIUDADE",
    "NATUREZA_FATO",
    "FATA_HORA_FATO",
    "CIDADE_FATO",
]

df = load_all_csvs("./input")
df_filtered = filter_by(df, values=["Maceió", "Arapiraca"], column_name="CIDADE_FATO")
df_out = select_columns(df_filtered, desired_columns)
save(df_out, output_path="./output/filtered_maceio_arapiraca.csv")

Saved 110644 rows to ./output/filtered_maceio_arapiraca.csv
